# Role attestations

Open this `.ipynb` in JupyterLab or an editor with Jupyter support. Install Deno's kernel with
`deno jupyter --install`, select **Deno**, and Run All from a fresh kernel. This notebook imports
exact published verifier **2.0.0**. It generates its own demonstration keys, asserts success and
refusal, prints only safe semantic outcomes and cleans up resources. The saved text is the expected
semantic output; CI executes each cell in the real kernel and compares it. Random key and signature
bytes are never persisted. See [runner instructions](README.md).

Verification is not authority. The host owns trusted inputs, replay persistence, revocation and
policy. The HTTP demonstration's memory reservations last one process; use the durable-store
recipe for a persistent host.


In [ ]:
import assertPackage from "node:assert/strict";
import verifierPackage from "npm:@bounded-authority-protocol/verifier@2.0.0/package.json" with { type: "json" };
import { BACKEND_ID as notebookBackend } from "npm:@bounded-authority-protocol/verifier@2.0.0";
assertPackage.equal(verifierPackage.version, "2.0.0");
assertPackage.equal(notebookBackend, "portable");
console.log("verifier 2.0.0; portable backend");


verifier 2.0.0; portable backend


In [ ]:
import assert from "node:assert/strict";
import { generateKeyPairSync, sign, type KeyObject } from "node:crypto";
import { roleAttestation, verifyGrant } from "npm:@bounded-authority-protocol/verifier@2.0.0";

const raw = (key: KeyObject): Uint8Array => Buffer.from(key.export({ format: "jwk" }).x!, "base64url");
const root = generateKeyPairSync("ed25519"); // the attestor, for example an organization's root key
const issuerKey = generateKeyPairSync("ed25519"); // the subject

// The root key attests that "issuer-2026-10" may act as an issuer from t=1000 until t=2000.
const input = roleAttestation.attestationSigningInput({
  keyId: "root-2026",
  jti: "attestation-0001",
  subjectKeyId: "issuer-2026-10",
  subjectPublicKey: raw(issuerKey.publicKey),
  role: "issuer",
  notBefore: 1_000,
  expiresAt: 2_000,
});
assert.ok(input.ok);
const message = Buffer.concat([input.value.protectedSegment, Buffer.from("."), input.value.payloadSegment]);
const attestation = roleAttestation.assembleAttestationCompact(input.value, sign(null, message, root.privateKey));
assert.ok(attestation.ok);

// The verifier supplies the attestor key it trusts (with its validity window), the subject it
// expects, and the time.
const expected = {
  attestor: { keyId: "root-2026", publicKey: raw(root.publicKey), validFrom: 0, validBefore: null },
  subjectKeyId: "issuer-2026-10",
  subjectPublicKey: raw(issuerKey.publicKey),
  now: 1_500,
};
const facts = roleAttestation.verifyAttestation(attestation.value, expected);
assert.ok(facts.ok);
assert.equal(facts.value.role, "issuer");
assert.equal(facts.value.verification, "signature_and_window");
assert.equal(facts.value.trust, "not_evaluated");

// The window is half-open: [notBefore, expiresAt).
assert.deepEqual(roleAttestation.verifyAttestation(attestation.value, { ...expected, now: 2_000 }), { ok: false });
// A different subject key under the same key id rejects.
assert.deepEqual(
  roleAttestation.verifyAttestation(attestation.value, { ...expected, subjectPublicKey: raw(generateKeyPairSync("ed25519").publicKey) }),
  { ok: false },
);
// An attestation is not a grant, and a grant verifier refuses it.
assert.deepEqual(
  verifyGrant(attestation.value, { keyId: "root-2026", publicKey: raw(root.publicKey) },
    { issuer: "root-2026", audience: "x", evaluationTime: 1_500, clockSkew: 0 }),
  { ok: false },
);

console.log("attestation: verified; expired and wrong subject refused");


attestation: verified; expired and wrong subject refused
